In [9]:
!pip install transformers torch requests beautifulsoup4 scikit-learn nltk -q

import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

url = "https://apnews.com/article/boeing-aviation-aircraft-air-india-crash-f12b20e65dc57ae655a1e0759b58938f"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
}

response = requests.get(url, headers=headers, timeout=15)
soup = BeautifulSoup(response.text, "html.parser")
paragraphs = soup.find_all("p")
article_text = " ".join([p.get_text() for p in paragraphs if len(p.get_text()) > 60])

print(f"Status code : {response.status_code}")
print(f"Characters  : {len(article_text)}")
print("\n--- PREVIEW ---")
print(article_text[:800])

Status code : 200
Characters  : 3067

--- PREVIEW ---
The Boeing logo is displayed at the company’s factory, Sept. 24, 2024, in Renton, Wash. (AP Photo/Lindsey Wasson, File) The crash of a Boeing 787 passenger jet in India minutes after takeoff on Thursday is putting the spotlight back on a beleaguered manufacturer though it was not immediately clear why the plane crashed.  The Air India 787 went down in the northwestern city of Ahmedabad with more than 240 people aboard shortly after takeoff, authorities said. It was the first fatal crash since the plane, also known as the Dreamliner, went into service in 2011, according to the Aviation Safety Network database. Boeing shares fell more than 4% in afternoon trading. The 787 was the first airliner to make extensive use of lithium ion batteries, which are lighter, recharge faster and can hold mo


## 1. Classify the sentiment, the intent, and the emotions.

In [10]:
from transformers import pipeline
chunks = [article_text[i:i+400] for i in range(0, len(article_text), 400)]

# ─────────────────────────────────────────────
# LLM METHOD — BART zero-shot
# ─────────────────────────────────────────────
print("\n── LLM METHOD: BART zero-shot (facebook/bart-large-mnli) ──")
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Sentiment
print("\n[LLM] Sentiment")
llm_sent = zsc(article_text[:1000],
               candidate_labels=["positive sentiment", "negative sentiment", "neutral sentiment"])
for label, score in zip(llm_sent["labels"], llm_sent["scores"]):
    print(f"  {label:<25} {'█' * int(score*40)} {score:.3f}")
llm_sent_verdict = llm_sent["labels"][0]

# Intent
print("\n[LLM] Intent")
llm_intent = zsc(article_text[:1000],
                 candidate_labels=["informational reporting", "investigative journalism",
                                   "corporate accountability", "breaking news", "opinion"])
for label, score in zip(llm_intent["labels"], llm_intent["scores"]):
    print(f"  {label:<30} {'█' * int(score*40)} {score:.3f}")
llm_intent_verdict = llm_intent["labels"][0]

# Emotion
print("\n[LLM] Emotion")
llm_emotion = zsc(article_text[:1000],
                  candidate_labels=["grief", "shock", "fear", "anger", "distrust", "neutral", "hope"])
for label, score in zip(llm_emotion["labels"], llm_emotion["scores"]):
    print(f"  {label:<20} {'█' * int(score*40)} {score:.3f}")
llm_emotion_verdict = llm_emotion["labels"][0]


# ─────────────────────────────────────────────
# DEEP LEARNING METHOD — fine-tuned models
# ─────────────────────────────────────────────
print("\n── DEEP LEARNING METHOD: Fine-tuned models ──")

# Sentiment — DistilBERT fine-tuned on SST-2
print("\n[DL] Sentiment — DistilBERT fine-tuned on SST-2")
dl_sent_pipe = pipeline("sentiment-analysis",
                        model="distilbert-base-uncased-finetuned-sst-2-english")
dl_sent_results = [dl_sent_pipe(chunk)[0] for chunk in chunks]

label_counts = {}
for r in dl_sent_results:
    lbl = r["label"]
    label_counts[lbl] = label_counts.get(lbl, 0) + 1

for label, count in label_counts.items():
    pct = count / len(chunks)
    print(f"  {label:<12} {'█' * int(pct*40)} {count}/{len(chunks)} ({pct*100:.1f}%)")
dl_sent_verdict = max(label_counts, key=label_counts.get)

# Emotion — DistilRoBERTa fine-tuned on Ekman emotions
print("\n[DL] Emotion — j-hartmann/emotion-english-distilroberta-base")
dl_emo_pipe = pipeline("text-classification",
                       model="j-hartmann/emotion-english-distilroberta-base",
                       top_k=None)

raw = dl_emo_pipe(article_text[:512])
if isinstance(raw[0], list):
    dl_emo_results = sorted(raw[0], key=lambda x: x["score"], reverse=True)
else:
    dl_emo_results = sorted(raw, key=lambda x: x["score"], reverse=True)

for item in dl_emo_results:
    print(f"  {item['label']:<12} {'█' * int(item['score']*40)} {item['score']:.3f}")
dl_emotion_verdict = dl_emo_results[0]["label"]

# Intent
print("\n[DL] Intent")
print("  No publicly available fine-tuned news-intent model exists.")
print("  Intent classification handled by LLM zero-shot only.")



print(f"\n  {'Task':<12} {'LLM (BART zero-shot)':<35} {'Deep Learning (fine-tuned)'}")
print(f"  {'-'*12} {'-'*35} {'-'*28}")
print(f"  {'Sentiment':<12} {llm_sent_verdict:<35} {dl_sent_verdict}")
print(f"  {'Emotion':<12} {llm_emotion_verdict:<35} {dl_emotion_verdict}")
print(f"  {'Intent':<12} {llm_intent_verdict:<35} {'N/A'}")


── LLM METHOD: BART zero-shot (facebook/bart-large-mnli) ──


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


[LLM] Sentiment
  negative sentiment        ████████████████████████████████████ 0.904
  neutral sentiment         ███ 0.081
  positive sentiment         0.014

[LLM] Intent
  breaking news                  ███████████████ 0.375
  corporate accountability       █████████████ 0.328
  informational reporting        █████ 0.146
  opinion                        ███ 0.083
  investigative journalism       ██ 0.067

[LLM] Emotion
  distrust             ██████████ 0.272
  fear                 ███████ 0.187
  grief                ██████ 0.170
  shock                ██████ 0.170
  anger                ████ 0.115
  neutral              ██ 0.072
  hope                  0.013

── DEEP LEARNING METHOD: Fine-tuned models ──

[DL] Sentiment — DistilBERT fine-tuned on SST-2


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  NEGATIVE     ███████████████████████████████████ 7/8 (87.5%)
  POSITIVE     █████ 1/8 (12.5%)

[DL] Emotion — j-hartmann/emotion-english-distilroberta-base


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  sadness      ████████████████████████████████████ 0.904
  fear         ██ 0.055
  surprise      0.016
  anger         0.010
  neutral       0.008
  joy           0.004
  disgust       0.002

[DL] Intent
  No publicly available fine-tuned news-intent model exists.
  Intent classification handled by LLM zero-shot only.

Q1 COMPARISON

  Task         LLM (BART zero-shot)                Deep Learning (fine-tuned)
  ------------ ----------------------------------- ----------------------------
  Sentiment    negative sentiment                  NEGATIVE
  Emotion      distrust                            sadness
  Intent       breaking news                       N/A


## 2. Determine how much the article is about technology, aviation, and policies.

In [11]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

# ─────────────────────────────────────────────
# LLM METHOD — BART zero-shot multi-label
# ─────────────────────────────────────────────
print("\n── LLM METHOD: BART zero-shot multi-label ──")

topic_labels = [
    "aviation and flight safety",
    "technology and aircraft engineering",
    "government policy and regulation"
]

llm_topics = zsc(article_text, candidate_labels=topic_labels, multi_label=True)
topic_scores = dict(zip(llm_topics["labels"], llm_topics["scores"]))
llm_top_topic = llm_topics["labels"][0]

for label, score in zip(llm_topics["labels"], llm_topics["scores"]):
    print(f"  {label:<42} {'█' * int(score*40)} {score:.3f}")


# ─────────────────────────────────────────────
# DEEP LEARNING METHOD — TF-IDF keyword scoring
# ─────────────────────────────────────────────
print("\n── DEEP LEARNING METHOD: TF-IDF keyword scoring ──")

topic_keywords = {
    "Aviation": [
        "crash", "takeoff", "mayday", "altitude", "fleet", "airline",
        "aircraft", "pilot", "runway", "aviation", "dreamliner",
        "flight", "airborne", "landing", "wingspan"
    ],
    "Technology": [
        "lithium", "battery", "787", "dreamliner", "flight data",
        "cockpit", "recorder", "fuel switch", "sensor", "electrical",
        "engineering", "software", "system", "composite", "hardware"
    ],
    "Policy": [
        "FAA", "NTSB", "DGCA", "AAIB", "regulator", "investigation",
        "directive", "inspection", "whistleblower", "mandate",
        "compliance", "advisory", "federal", "authority", "congress"
    ]
}

sentences = sent_tokenize(article_text)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)
feature_names = list(vectorizer.get_feature_names_out())
tfidf_array = tfidf_matrix.toarray()

topic_tfidf_scores = {}
for topic, keywords in topic_keywords.items():
    score = 0
    for kw in keywords:
        kw_lower = kw.lower()
        if kw_lower in feature_names:
            idx = feature_names.index(kw_lower)
            score += tfidf_array[:, idx].sum()
    topic_tfidf_scores[topic] = score

total_score = sum(topic_tfidf_scores.values())
dl_top_topic = max(topic_tfidf_scores, key=topic_tfidf_scores.get)

print(f"\n  TF-IDF topic scores (normalized):")
for topic, score in topic_tfidf_scores.items():
    pct = score / total_score if total_score > 0 else 0
    print(f"  {topic:<12} {'█' * int(pct*40)} {pct*100:.1f}%")

print(f"\n  Keywords matched per topic:")
for topic, keywords in topic_keywords.items():
    found = [k for k in keywords if k.lower() in article_text.lower()]
    print(f"  {topic:<12} ({len(found)}/{len(keywords)}): {', '.join(found)}")

avi_llm  = topic_scores.get("aviation and flight safety", 0)
tech_llm = topic_scores.get("technology and aircraft engineering", 0)
pol_llm  = topic_scores.get("government policy and regulation", 0)
avi_dl   = topic_tfidf_scores.get("Aviation", 0)  / total_score if total_score else 0
tech_dl  = topic_tfidf_scores.get("Technology", 0) / total_score if total_score else 0
pol_dl   = topic_tfidf_scores.get("Policy", 0)    / total_score if total_score else 0



print(f"\n  {'Topic':<12} {'LLM score':<18} {'TF-IDF score'}")
print(f"  {'-'*12} {'-'*18} {'-'*14}")
print(f"  {'Aviation':<12} {avi_llm:.3f}{'':>12} {avi_dl*100:.1f}%")
print(f"  {'Technology':<12} {tech_llm:.3f}{'':>12} {tech_dl*100:.1f}%")
print(f"  {'Policy':<12} {pol_llm:.3f}{'':>12} {pol_dl*100:.1f}%")
print(f"\n  LLM top topic    : {llm_top_topic}")
print(f"  TF-IDF top topic : {dl_top_topic}")


── LLM METHOD: BART zero-shot multi-label ──
  aviation and flight safety                 ██████████████████████████████████████ 0.974
  technology and aircraft engineering        ████████████████████████████████████ 0.919
  government policy and regulation           ██████████████████████████ 0.664

── DEEP LEARNING METHOD: TF-IDF keyword scoring ──

  TF-IDF topic scores (normalized):
  Aviation     ██████████████████████ 55.5%
  Technology   █████████████████ 44.5%
  Policy        0.0%

  Keywords matched per topic:
  Aviation     (8/15): crash, takeoff, fleet, airline, aircraft, pilot, aviation, dreamliner
  Technology   (5/15): lithium, 787, dreamliner, sensor, system
  Policy       (1/15): regulator

  Topic        LLM score          TF-IDF score
  ------------ ------------------ --------------
  Aviation     0.974             55.5%
  Technology   0.919             44.5%
  Policy       0.664             0.0%

  LLM top topic    : aviation and flight safety
  TF-IDF top topic : A

## 3. Use LLMs and one Deep Learning method of your choice to answer the questions. Compare the results.


In [13]:
print("=" * 60)
print("Q3 — FULL COMPARISON: LLM vs DEEP LEARNING")
print("=" * 60)

print(f"""
METHODS USED
------------
  LLM Method : BART-large-MNLI (zero-shot, no fine-tuning needed)
  DL Method  : DistilBERT-SST2 (sentiment) + DistilRoBERTa-Ekman (emotion)

Q1 — SENTIMENT
--------------
  LLM (BART zero-shot) : {llm_sent_verdict}
  DL  (DistilBERT)     : {dl_sent_verdict}
  Agreement            : {'✅ Both agree' if llm_sent_verdict.split()[0].lower() == dl_sent_verdict.lower() else '❌ Disagree'}

Q1 — EMOTION
------------
  LLM (BART zero-shot)       : {llm_emotion_verdict} (score: {llm_emotion["scores"][0]:.3f})
  DL  (DistilRoBERTa Ekman)  : {dl_emotion_verdict}  (score: {dl_emo_results[0]["score"]:.3f})

Q1 — INTENT
-----------
  LLM (BART zero-shot) : {llm_intent_verdict} (score: {llm_intent["scores"][0]:.3f})
  DL                   : No fine-tuned model available — LLM only

Q2 — TOPIC COMPOSITION
-----------------------
  Topic        LLM (BART)    DL (TF-IDF)
  Aviation     {avi_llm:.3f}         {avi_dl*100:.1f}%
  Technology   {tech_llm:.3f}         {tech_dl*100:.1f}%
  Policy       {pol_llm:.3f}         {pol_dl*100:.1f}%

  LLM top topic    : {llm_top_topic}
  TF-IDF top topic : {dl_top_topic}

CONCLUSIONS
-----------
  1. Sentiment  : Both methods agree — article is overwhelmingly negative
                  due to mass casualties and Boeing safety concerns.

  2. Emotion    : LLM labeled '{llm_emotion_verdict}', DL labeled '{dl_emotion_verdict}'.
                  Difference reflects label schemas, not true disagreement —
                  both point to distress-category emotions.

  3. Intent     : LLM zero-shot is the only viable method here.
                  Fine-tuned intent classifiers for news do not exist publicly.

  4. Topics     : Both rank Aviation as dominant. LLM captures semantic
                  meaning; TF-IDF relies on keyword frequency alone.

  5. Overall    : LLM zero-shot is more flexible — works across all tasks
                  with no retraining. Fine-tuned DL models are faster and
                  more precise but limited to their specific trained task.
""")

Q3 — FULL COMPARISON: LLM vs DEEP LEARNING

METHODS USED
------------
  LLM Method : BART-large-MNLI (zero-shot, no fine-tuning needed)
  DL Method  : DistilBERT-SST2 (sentiment) + DistilRoBERTa-Ekman (emotion)

Q1 — SENTIMENT
--------------
  LLM (BART zero-shot) : negative sentiment
  DL  (DistilBERT)     : NEGATIVE
  Agreement            : ✅ Both agree

Q1 — EMOTION
------------
  LLM (BART zero-shot)       : distrust (score: 0.272)
  DL  (DistilRoBERTa Ekman)  : sadness  (score: 0.904)

Q1 — INTENT
-----------
  LLM (BART zero-shot) : breaking news (score: 0.375)
  DL                   : No fine-tuned model available — LLM only

Q2 — TOPIC COMPOSITION
-----------------------
  Topic        LLM (BART)    DL (TF-IDF)
  Aviation     0.974         55.5%
  Technology   0.919         44.5%
  Policy       0.664         0.0%

  LLM top topic    : aviation and flight safety
  TF-IDF top topic : Aviation

CONCLUSIONS
-----------
  1. Sentiment  : Both methods agree — article is overwhelmingl